**Team #18**

| Student ID | Name | Email |
|---|---|---|
| 20258019 | Dongjin Kim | dj_kim@kaist.ac.kr |
| 20258212 | Hoiyeong Jin | hy.jin@kaist.ac.kr |
| 20265280 | Seokju Yun | sj_yun@kaist.ac.kr |

---

# Teaching a Robot to Learn From Its Own Mistakes
### A from-scratch tutorial on RECAP (π\*0.6), rebuilt on LeRobot π0.5 + RoboCasa

Imagine you are teaching someone to cook by only ever letting them *watch* videos of a chef.
They might get pretty good — but the first time they knock over a pan, they have no idea what
to do, because no video ever showed a knocked-over pan. To truly improve, they need to *try
things themselves*, see what goes wrong, and learn from it.

That gap — between *copying an expert* and *learning from your own experience* — is the whole
story of this notebook. We will:

1. Explain, from zero, what a **robot-controlling AI (a “VLA”)** is.
2. Meet **RoboCasa**, a kitchen simulator where our robot practices.
3. Build the standard “just copy the expert” baseline (**behavior cloning / SFT**).
4. Build **RECAP**, the “learn from your own experience” method from the π\*0.6 paper.
5. Ask our own research question and report the result: with a **scene-aware value function**
   and the **right advantage threshold (ε = 0.5)**, our RECAP policy **matches SFT on
   OpenDrawer and surpasses it on the harder PnP task** — i.e. equal-or-better than plain
   copying — and we show *why the threshold ε is the make-or-break knob* at this scale.

---

**Who is this for?** Anyone curious about modern robot learning. **No prior knowledge of
VLAs, the π (“pi”) model series, RoboCasa, or reinforcement learning is assumed.** If you
can read a little Python, you can read this.

**This notebook is self-contained.** It is distributed as a single `.ipynb` file. The very
first cell fetches the companion GitHub repo
([k00dj-19/pistar0.6-recap-robocasa](https://github.com/k00dj-19/pistar0.6-recap-robocasa))
— which holds the diagrams, the robot videos, and all the training/eval code — so everything
below renders and runs. **Just run the cells top to bottom** (in Colab: *Runtime → Run all*).

**Two ways to read:**
- **Path 1 — no GPU needed (just read).** Run the first **setup** cell to pull in the figures
  and robot videos, then scroll through. Every table and result is already filled in.
- **Path 2 — with a GPU (run it yourself).** Flip `RUN_EVAL = True` in the config cell to
  download our trained robot brains from the HuggingFace Hub and watch them perform in the
  simulator. Instructions are at the very end.

> **Honesty note, up front.** The original π\*0.6 paper (Physical Intelligence,
> [arXiv:2511.14759](https://arxiv.org/abs/2511.14759)) reports impressive *real-world*
> results (folding laundry, making espresso) using private data and model weights that are
> not public. We could not reproduce *that*. Instead we did a faithful **re-implementation**
> of the method in **simulation**, on the closest open model (π0.5), and we report what
> actually happened — including where it fell short. This is a careful study, not a victory lap.

## Setup (run me first): fetch the project files
This notebook ships as a single file. The cell below clones the companion GitHub repo and
steps into it, so every figure, video, and script below is available. It is safe to re-run —
it clones only once, and if you are *already* running from inside a checkout of the repo it
simply does nothing.

In [ ]:
# --- bootstrap: make this single-file notebook self-contained ---
import os, sys, subprocess

REPO_URL = 'https://github.com/k00dj-19/pistar0.6-recap-robocasa.git'
REPO_DIR = 'pistar0.6-recap-robocasa'

def _inside_repo():
    # we are 'inside the repo' once these project folders are visible from the cwd
    return os.path.isdir('docs/assets') and os.path.isdir('recap')

if not _inside_repo():
    if not os.path.isdir(REPO_DIR):
        print('Cloning companion repo (one time) ...')
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

sys.path.insert(0, os.getcwd())   # make the repo's `recap` package importable
assert _inside_repo(), f'bootstrap failed: project files not found in {os.getcwd()}'
print('working directory:', os.getcwd())
print('repo contents     :', sorted(os.listdir('.')))

## Setup: one small config cell
Run this first. It just sets a few knobs and checks that the figures/videos folder is
present. Nothing here needs a GPU.

In [ ]:
# --- config ---
RUN_EVAL = False           # leave False to just read (Path 1). Set True on a GPU (Path 2).
SEED, N_EPISODES, N_ACTION_STEPS = 5000, 50, 10   # the exact eval settings behind every number below
# local checkpoint root (only used if you trained locally; the HF Hub is the default for Path 2)
CKPT_ROOT = 'outputs/robocasa/specialist_v2'
ASSETS = 'docs/assets'
import os; print('assets folder:', os.listdir(ASSETS) if os.path.isdir(ASSETS) else 'MISSING')

## 1. The big picture: why is teaching robots so hard?

Getting a robot arm to do everyday physical chores — open a drawer, put a mug away, close
the fridge — is one of the hardest problems in AI. A chatbot only has to produce *text*. A
robot has to produce *motion*, in the real, messy, physical world, where a few centimeters
or a fraction of a second is the difference between success and a dropped cup.

### Imitation learning: copy the expert
The most popular recipe today is **imitation learning**. A human teleoperates the robot
to do a task correctly many times. We record those demonstrations — *“when the robot saw
this, the human moved it like that”* — and we train a neural network to **copy** the human.
This is also called **behavior cloning** or **supervised fine-tuning (SFT)**, and it works
remarkably well.

### The built-in ceiling
But copying has a hard ceiling, for an intuitive reason: **the robot only ever sees the
expert doing things *right*.** So:
- It never learns to **recover** from a mistake — because the expert never made one on camera.
- It gets confused in any situation the demos didn’t cover.
- It can **never become better than the demonstrations** — a perfect copy of a B+ chef is
  still a B+ chef.

### Learning from experience
Humans break through this ceiling by **practicing**: we try, we fail, we notice what went
wrong, and we adjust. The dream is to give robots the same ability — to let a robot run on
its own, collect its *own* attempts (most of them mediocre), figure out which moments were
good and which were bad, and then **learn from that experience** to improve past the demos.

That is exactly what the **RECAP** method (the heart of the π\*0.6 paper) tries to do, and
it is what this notebook builds and tests.

## 2. What is a VLA? And what is this “π” series?

### VLA = Vision-Language-Action model
A **VLA** is a single neural network that ties together three things:
- **Vision** — it looks at camera images of the scene.
- **Language** — it reads a natural-language instruction (“open the drawer”).
- **Action** — it outputs the actual motor commands to move the robot.

A useful analogy: a VLA is like a **chauffeur who understands spoken directions**. You say
“pull into the third driveway,” they *see* the street through the windshield, *understand*
your words, and *act* by steering and braking — all as one fluid skill, not three separate
programs bolted together.

![What a VLA is](docs/assets/diagram_vla.png)

### The π (“pi”) series from Physical Intelligence
“π” (pi) is a family of VLAs built by the company **Physical Intelligence**. Each version
builds on the previous one:

| Model | One-line idea |
|---|---|
| **π0** | A VLA that produces smooth, continuous motion in *chunks* using **flow matching** (explained below), on top of a vision-language model. |
| **π0.5** | Adds **open-world generalization** — trained on very diverse data so it copes with new homes/objects, using a trick called *knowledge insulation*. **This is the open model we build on** (weights: [`lerobot/pi05_base`](https://huggingface.co/lerobot/pi05_base)). |
| **π\*0.6** | The paper we re-implement ([arXiv:2511.14759](https://arxiv.org/abs/2511.14759)). Adds **RECAP**: learning from the model’s *own experience* plus human *corrections*. **No public weights**, so we port its method onto π0.5. |

### What’s inside π0.5 (the model we actually use)
You do not need to memorize this, but it helps to picture the two halves (both drawn in the
diagram above):
1. A **PaliGemma backbone** — an off-the-shelf vision-language model = a **SigLIP** image
   encoder (the “eyes”) + a **Gemma** language model (the “reading” brain).
2. A separate **action expert** — a smaller network that turns the backbone’s understanding
   into an **action chunk**: 50 future steps of motion, predicted at once.

Each timestep, π0.5 is fed **3 camera images + the robot’s state (16 numbers) + the text
prompt**, and it outputs that chunk of motion.

> **Why you’ll see “50” and “10.”** The model *predicts* a 50-step chunk at once, but at run
> time we only *execute the first 10 steps* and then re-plan from fresh observations (a common
> trick called *receding-horizon* control). That is the `n_action_steps = 10` you’ll see in
> the eval config — predict 50, act on 10, repeat.

> **What is “flow matching”? (high level)** It is a way to *generate* continuous outputs
> (here, smooth motions) by starting from random noise and **iteratively cleaning it up**
> into a coherent action — the same family of ideas as the *diffusion* models behind AI image
> generators. You can think of it as “sculpting motion out of static.” That is all you need
> for this tutorial.

## 3. Meet RoboCasa: our robot’s practice kitchen

Training real robots is slow, expensive, and breakable. So, like flight schools use flight
simulators, robot researchers use **physics simulators**. We use **RoboCasa**, a large
benchmark for **kitchen manipulation** built on the **robosuite** framework and the
**MuJoCo** physics engine. It simulates realistic kitchens, objects, and a robot you can
command and measure precisely.

**The robot — “PandaOmron”:** a **Franka Panda** arm (a popular 7-jointed research arm)
mounted on a wheeled **Omron** mobile base. Think *arm on a cart*.

**What the robot senses each step (its “observation”):**
- **3 RGB cameras** at 256×256: a left and right `agentview` (looking at the scene) and an
  `eye_in_hand` camera (mounted on the gripper, for close-up work).
- A **16-dimensional proprioceptive state** — “proprioception” just means the robot’s sense
  of *its own body*: joint angles, gripper width, base position, etc.

**What the robot does each step (its “action”):** a **12-number** command (arm motion,
gripper, base motion), issued **20 times per second** (20 fps).

### The two tasks we study
We focus our study on **two contrasting kitchen tasks**, chosen to span the difficulty range
— one where behavior cloning is already strong (little room to improve) and one hard task
where it leaves clear headroom. Watch one demonstration of each below — this is the kind of
expert behavior our robot will first learn to copy:

| Task | Goal | Why we picked it |
|---|---|---|
| **OpenDrawer** | Pull a drawer open. | A solid mid-difficulty task where SFT is already strong (58%) — a hard bar to clear. |
| **PickPlaceCounterToCabinet** (“PnP”) | Pick an object off the counter and place it in the cabinet. | The **hardest** task (it chains grasping *and* placing); SFT is weakest here (36%), so there is the **most headroom** for learning-from-experience to help. |

In [ ]:
# Demonstration of each task (what 'success' looks like). No GPU needed — just plays the clips.
from pathlib import Path
from IPython.display import Video, display, Markdown
GALLERY = f'{ASSETS}/videos'
DEMO_TASKS = ['OpenDrawer', 'PickPlaceCounterToCabinet']
shown = False
for t in DEMO_TASKS:
    p = Path(GALLERY) / f'demo_{t}.mp4'
    if p.exists():
        display(Markdown(f'**Demonstration — {t}**')); display(Video(str(p), embed=True, width=360))
        shown = True
if not shown:
    print('demo videos not staged yet — expected at', GALLERY, '(demo_<Task>.mp4)')

### What does “success” mean, and what is “closed-loop eval”?
- **Success** is a simple **yes/no check built into the simulator** — e.g. “is the fridge
  door angle below X degrees?” It is objective and automatic; no human judging.
- **Closed-loop evaluation** means we let the policy actually *drive the robot* in the
  simulator for a whole episode, reacting to what happens moment-to-moment (a “closed loop”
  between seeing and acting), and check success at the end. We do this for **50 episodes**
  on a **held-out random seed (5000)** the model never trained on, and report the **% that
  succeed**. That single percentage is how we score every method in this notebook.

## 4. The baseline: behavior cloning (SFT), explained in detail

**Supervised Fine-Tuning (SFT)** here is exactly **behavior cloning**: we train the policy to
**imitate the demonstration action sequences**. The recipe is almost embarrassingly simple:

1. Collect demonstrations: many `(what the robot saw, what the expert did next)` pairs.
2. Show the policy the “saw” part and ask it to predict the “did” part.
3. Nudge the network whenever its prediction differs from the expert’s actual move.
4. Repeat over all the data until it reliably reproduces expert-like motion.

It is the robotics version of **“watch and copy.”** And it is a *strong* baseline — do not
underestimate it. With good demos, behavior cloning alone solves a lot.

### But remember the ceiling (§1)
Because SFT only ever sees the expert doing things correctly:
- **No recovery skills.** If the policy drifts into a state the expert never visited (because
  the expert never fumbled), it has no idea what to do — errors can snowball.
- **No coverage beyond the demos.** Unusual situations = guesswork.
- **No way to exceed the demos.** Copying a demonstrator caps you *at* the demonstrator.

In this notebook, **SFT is our baseline** — the bar that the fancier “learn from experience”
method (RECAP) has to clear. Keep these limits in mind; they are the entire motivation for
what comes next.

## 5. RECAP: learning from your own experience

**RECAP** = *RL with Experience and Corrections via Advantage-conditioned Policies* (**RL** =
*reinforcement learning* — learning by trial-and-error rather than by copying). Don’t worry
about the acronym — here is the whole idea through one analogy, then the four moving parts.

### The coach analogy
Imagine a **basketball coach reviewing game film** with a young player. The coach watches
every clip and labels each one: **“good play”** or **“bad play.”** Crucially, the player
studies *both* — not just the highlights — so they learn what good looks like *and* what to
avoid. Then, in the next game, the coach simply says: **“play the good way.”**

RECAP does exactly this for a robot. It lets the policy generate its own attempts, has a
“coach” label each moment good or bad, trains the policy on *both* kinds (each clearly
tagged), and at game time just asks for the good kind.

![The RECAP loop](docs/assets/diagram_recap_loop.png)

### The four moving parts

**1. A value function — the coach’s eye.** This is the heart of the method, so let’s be
concrete. The **value function** `V(o)` is a *separate, small* network (it is **not** the
robot policy). It looks at a single moment `o` (`o` = *observation* — one moment’s worth of
what the robot sees) and answers one question: **“starting from here, how well is this
attempt going?”** Its inputs and output:

- **Input — what it sees.** A summary of the moment plus *which task* is being attempted.
  For our **scene-aware** critic (§7) the “what it sees” is not raw pixels but the VLA’s own
  **frozen visual features** — *frozen* means we reuse the VLA’s vision part exactly as-is
  (we don’t retrain it), and a *feature* / *embedding* is just a fixed-length list of numbers
  that summarizes an image: here a **2048-number** PaliGemma embedding of the three camera
  views, so the coach literally sees the scene the robot sees. A small **task embedding** (a
  few learned numbers per task) tells it whether this is OpenDrawer or PnP. A small standard
  neural net (a 3-layer **MLP**, multi-layer perceptron) maps that to the output.
- **Output — a score on a fixed scale.** We define the scale by the eventual **outcome** of
  the trajectory the moment came from, normalized to **[−1, 0]**: a moment on a path that
  ends in **success scores near 0**; a moment on a path that **fails scores near −1**. So
  `V(o) ≈ 0` means *“on track,”* `V(o) ≈ −1` means *“this is going badly.”*
- **How it’s trained (lightly).** The training target for a frame is its **return** — here
  simply *whether the episode that frame belongs to eventually succeeded (→ near 0) or failed
  (→ near −1)*; “**Monte-Carlo**” just means we read that off the actual finished episode
  rather than estimating it. Rather than predict one number, `V(o)` predicts a **probability
  spread over 201 little bins** between −1 and 0 (think: “70% chance this ends badly, 30% ok”)
  and we report its mean. Predicting a spread instead of a point is simply more stable to
  train. (Full equation in the “under the hood” box.)

Does it actually work? The two panels below — both on **held-out data the critic never
trained on** — say yes:

![Value function: value-over-time and calibration](docs/assets/fig4_value_function.png)

The figure has **one row per task** (OpenDrawer on top, PnP on the bottom), each with two
panels:
- **Left — value over time.** Each line is one episode, with `V(o)` plotted from start
  (time 0) to end (time 1). **Green solid lines are successful episodes**: their value
  **rides near 0** — the coach can *feel* the attempt going well. **Red dashed lines are
  failures**: they **sit down near −1**. The two families clearly separate, which is exactly
  what we need to tell good moments from bad.
- **Right — calibration.** Every dot is one held-out moment: its **true** outcome-return
  (x-axis) vs the critic’s **predicted** `V(o)` (y-axis). The red line is perfect prediction
  (`y = x`); dots clustering along it mean the critic is **calibrated** — its numbers match
  reality, not merely rank moments in the right order. (Points pile up near 0 simply because
  most frames come from successful episodes.)

**2. The advantage — “did this stretch go *better than expected*?”** From the value function
we compute an **advantage** `A(o_t)` for each moment `t` (a *timestep*): looking ~50 steps
ahead, did things improve *more than* the value function expected at `t`? *Worked example:*
if the coach scored the moment at −0.4 and 50 steps later the situation is worth −0.1, that
**+0.3 better-than-expected** swing is the advantage. Positive advantage = that bit of
behavior was genuinely good; negative = it made things worse.

**3. Binarize — turn the score into a clean label.** **ε (epsilon) is a *fraction*:** we keep
the **top-ε fraction** of moments (by advantage) as **“positive”** and call the rest
**“negative.”** So **ε = 0.5 labels the best-scoring 50% of moments positive**; ε = 0.1, only
the top 10%. The histogram below shows the advantage distribution for each task with the
**ε = 0.5 cutoff** (dark line) — our main setting (§10). Advantages cluster tightly around 0
(most moments are unremarkable), so the cutoff sits just left of the peak; everything to its
right becomes a “positive” training example.

![Advantage distribution with the ε=0.5 cutoff](docs/assets/advantage_histogram.png)

**4. Condition the policy — tell it which kind to learn.** This is the clever, simple trick.
For every training frame we **append a phrase to the language prompt**: either
`Advantage: positive` or `Advantage: negative` (and 30% of the time we drop it, so the
policy also works with no tag). The policy thus learns to produce *both* good and bad
behavior — each labeled. **At inference we simply prompt `Advantage: positive`** to summon
the good kind. No new network architecture — just a smarter prompt.

### Putting it in a loop (the paper’s main loop, repeated K = 3 times)
The four items above are the *ingredients*; now we use them in a repeating *loop* (here **K**
= how many times we repeat it; we use K = 3). RECAP repeats the four loop steps in the diagram:
1. **Roll out** the current policy in the simulator to gather fresh experience.
2. **Label** every new frame with the value function (positive / negative).
3. **Add** it to the growing data pool.
4. **Retrain** the policy from the pretrained base on the whole pool, with the tags.

Repeat **K = 3** times — each round, the policy should (in principle) get a little better,
because it keeps practicing and re-learning from richer, labeled experience.

<details><summary><b>Under the hood (optional equations — safe to skip)</b></summary>

- **Distributional value function.** Instead of predicting a single number, `V(o)` predicts a
  **categorical distribution over 201 return bins**, trained with **cross-entropy** against
  the (normalized) Monte-Carlo return of the trajectory. Distributional targets are more
  stable to train than a single regressed scalar. Successful demos push the return toward 0;
  failed rollouts collapse it toward −1.
- **Advantage.** We use an **N-step estimate (N = 50)**: roughly, the value N steps later
  (plus rewards along the way) minus the value now — a measure of whether the stretch beat
  expectations.
- **Threshold ε.** Applied **per task** so each task gets a balanced set of positive frames.
- **Conditioning dropout.** The 30% prompt-drop lets the same network serve as both a
  conditioned and an unconditioned policy.

</details>

The conditioning is **literally a dataset-level prompt edit** (`recap/recap/advantage_dataset.py`) — there is *no* architecture change. The tag is just
appended to each frame's instruction text and tokenized like any other words:

```python
POS, NEG = ' Advantage: positive', ' Advantage: negative'
# per frame: positive if the advantage indicator says so, else negative;
#            dropped (unconditional) with probability `dropout` (= 0.30)
task = strip_advantage_phrase(item['task'])           # remove any phrase already baked in
item['task'] = task + (POS if is_positive else NEG)   # -> tokenized like any other prompt text
```

## 6. Our re-implementation: π0.5 + RoboCasa

With the concepts in hand, here is exactly what we built and the scope we chose:

- **Base model:** **π0.5** (open weights [`lerobot/pi05_base`](https://huggingface.co/lerobot/pi05_base)) — because there are **no
  public π0.6 weights**. We port the RECAP *method* onto π0.5.
- **Environment:** **RoboCasa in simulation** — because the paper’s real-world data and
  robots are private. Simulation also lets us measure success cleanly and cheaply.
- **Pipeline (the paper’s Algorithm 1, scaled down):**
  1. **Multi-task BC pretrain** — one base policy `π_pre` trained by behavior cloning across
     several RoboCasa kitchen tasks together.
  2. **Per-task specialist** — fine-tune `π_pre` into a focused expert for each task.
  3. **K = 3 RECAP** — run the experience loop (§5) on top, per task.

We report on the **two tasks introduced in §3** (OpenDrawer and PnP), chosen to span the
difficulty range.

The SFT baseline and each RECAP policy **start from the same `π_pre` and use the same
per-run fine-tuning budget** (same optimizer steps and batch size); they differ in *data +
conditioning*. So the final-stage training is matched. To be clear, RECAP’s *total* pipeline
is heavier — it also collects rollouts, trains a value function, and re-trains K = 3 times —
so “equal compute” means the **policy fine-tune step**, not the whole pipeline. (If anything,
that makes RECAP’s mere *parity* with SFT a conservative result: it spends more to get there.)

> **Scope honesty (again, because it matters):** this is a **faithful method re-implementation
> in a down-scaled setting**, not a reproduction of the paper’s real-world results. We will
> not imply otherwise.

## 7. Our research question: is the *coach* the bottleneck?

Here is our own contribution on top of the re-implementation. Recall that the whole method
hinges on the **value function** — it is the “coach” that decides which moments are labeled
good vs bad. **If the coach has poor judgment, every label is noisy, and RECAP learns from
bad labels.** So we asked:

> **Does the *quality* of the value function bottleneck RECAP at small scale?**

To test this, we compare two coaches:
- **Proprioception-only critic (the cheap coach).** Sees only the robot’s 16-number body
  state — **blind to the scene** (it cannot see the fridge, the mug, the drawer).
- **Scene-aware VLM value function (the smart coach).** A **VLM** (*vision-language model*) is
  the seeing-and-reading half of a VLA, without the action part. This critic is built on the
  VLA itself: it uses **frozen PaliGemma features** (the model’s own visual understanding), so
  it actually *sees* what the robot sees. This matches the paper’s scene-aware `V_pre` (the
  paper’s name for its pretrained value function).

If the scene-aware coach gives better labels and a better policy, then yes — the value
function was a real bottleneck. Let’s look at the numbers.

## 8. Main result: RECAP (scene-aware VLM critic, ε = 0.5) **matches or beats SFT**

Closed-loop success rate (**%, n = 50 episodes, held-out seed 5000**). We use our scene-aware
VLM value function (§7) and the advantage threshold **ε = 0.5** (justified in §10), and show
the SFT baseline alongside **each of the three RECAP iterations**. We compare against the
**final iteration (i3)** — the natural endpoint of the loop, fixed in advance (not the
best-looking iteration). i3 is in **bold**.

| Task | SFT (baseline) | RECAP i1 | RECAP i2 | **RECAP i3 (final)** | i3 vs SFT |
|---|---:|---:|---:|---:|:--:|
| OpenDrawer | 58 | 54 | 44 | **58** | = tie |
| PnP (CounterToCab) | 36 | 32 | 28 | **38** | +2 |

![RECAP iteration curve at ε=0.5](docs/assets/curves_eps50.png)

### How to read this
- **By the final iteration, RECAP reaches equal-or-better than SFT on both tasks:** it
  **ties** the strong SFT baseline on **OpenDrawer (58 = 58)** and **edges ahead** on the
  harder **PnP (38 vs 36)** — the task with the most headroom. With a scene-aware critic and a
  well-chosen ε, learning-from-experience **clears the behavior-cloning bar** here, rather
  than falling below it.
- **The gain concentrates where there is headroom.** On PnP (SFT only 36) RECAP adds value;
  on OpenDrawer (SFT already 58, near this setup’s ceiling) it matches but cannot exceed an
  essentially saturated task.
- **How big is the win, really? (read this honestly).** Each number is a proportion over
  **n = 50** episodes, so its 95% confidence interval is roughly **±13–14 points** (error bars
  on the plot). At this sample size, **none of the iteration-to-iteration or +2 differences
  are statistically significant** — the defensible claim is *“RECAP reaches parity with SFT
  (it does not degrade), with a small PnP gain that points the right way but sits within
  noise,”* **not** *“RECAP dominates.”* That parity is itself the result: at the paper’s
  default threshold RECAP falls clearly below SFT (§10), and closing that gap took the two
  ingredients below.
- **The curve is non-monotonic** (a dip at iter 2 before the iter-3 recovery), which we
  attribute to mixing in *off-policy* data — older rollouts from earlier policies (explained
  in §11).

Two ingredients got us to parity — the **scene-aware critic** (§9) and the **threshold ε**
(§10). We examine each next.

## 9. Ingredient 1 — did the coach matter? Critic ablation (proprio → VLM)

An **ablation** means: change exactly *one* component and see what moves. Here we hold
everything fixed and swap only the value function — the **cheap, scene-blind**
proprioception-only critic versus our **scene-aware VLM** critic.

> **Note on the numbers below:** this experiment was run at the **default ε = 0.3** (it
> predates the ε sweep in §10), so the absolute success rates are *lower* than §8’s ε = 0.5
> headline. That is expected — here we only compare **proprio vs VLM**, not vs SFT. We report
> the **final iteration (i3)**, the same metric as §10 (so the VLM/ε = 0.3 row matches §10).

| Critic (value function) | OpenDrawer (i3) | PnP (i3) |
|---|---:|---:|
| RECAP, proprioception-only | 46 | 28 |
| RECAP, **scene-aware VLM** | **46** | **32** |

**The finding — two parts, stated honestly:**
- **What we can *measure* directly: better labels.** The scene-aware critic is visibly
  better *calibrated* than a blind one — that is the value-over-time + calibration plot in
  §5 (`fig4`), which is held-out and unambiguous. A critic that sees the scene simply judges
  moments more accurately, which is the whole point of upgrading the coach.
- **Downstream policy effect: positive but within noise.** The resulting policy **ties on
  OpenDrawer (46 = 46)** and is **+4 on PnP (32 vs 28)** — never worse, but at n = 50 (±~14
  pts) this is not a statistically significant policy gain. So we **adopt the scene-aware
  critic on principle** (better labels, parity-or-better policy, and parity with the paper’s
  scene-aware `V_pre`), not on the basis of a measured win.

A better coach alone does not clear SFT at ε = 0.3 — for that we also needed the right
threshold, the second ingredient.

## 10. Ingredient 2 — the advantage threshold ε is the make-or-break knob

Recall ε decides what fraction of the most-advantaged moments get labeled **“positive.”** It
is the single most important RECAP hyperparameter, and the paper tunes it per task. We swept
**ε ∈ {0.1, 0.3, 0.5}** on the scene-aware VLM specialists (final iteration, success %, n = 50,
seed 5000):

| Task | ε = 0.1 | ε = 0.3 | ε = 0.5 |
|---|---:|---:|---:|
| OpenDrawer | 52 | 46 | **58** |
| PnP (CounterToCab) | 38 | 32 | **38** |

### The key finding: ε = 0.3 (the default) is the *worst* of the three — a U-shape
- For **both** tasks, the common default **ε = 0.3 is the lowest-scoring setting** (OpenDrawer
  46, PnP 32), and moving ε *either* direction helps — a **U-shape with its valley at
  ε = 0.3**. The best of the three points tested is **ε = 0.5** (OpenDrawer 58, PnP 38), which
  is why we use it in §8.
- **Scope caveat (important):** we tested only **{0.1, 0.3, 0.5}**, and **ε = 0.5 is the
  largest value we tried** — so we can say it is *the best within the tested range*, but **not**
  that it is a global optimum or “sweet spot”; we did not probe ε > 0.5, so the trend could
  keep rising. And with n = 50 (±~14 pts) the 52↔46↔58 wiggle is not individually
  significant — the robust, repeated-across-both-tasks signal is simply *“ε = 0.3 was the
  worst choice we tested.”*
- The intuition: too large an ε floods training with mediocre “positive” frames; too small
  starves it of positives — so an unlucky middle default can underperform the extremes.
- **This is the lever behind the §8 result.** The earlier impression that *“RECAP can’t beat
  behavior cloning”* was, at least in part, an artifact of evaluating at a poor default
  threshold — not a fixed property of the method. Pick a better ε and RECAP reaches parity
  (§8).

_(SFT is intentionally omitted from this table — it is a RECAP-internal hyperparameter sweep;
the head-to-head comparison against SFT lives in §8.)_

## 11. Discussion: why the win is real but modest (and the curve bumpy)

RECAP reaches **equal-or-better than SFT** (§8) — but the margin is small and the iteration
curve dips before it peaks. Both are exactly what we’d expect from running a method built for
a **large-scale, real-world** regime inside a **small-scale, clean-demo simulation**. Four
structural differences explain why the gains aren’t larger:

1. **Scale & diversity.** The paper uses thousands of hours across many skills; we use a
   handful of tasks. RECAP helps most exactly when behavior cloning is **data-starved** — so
   the biggest gains show up on our **hardest, lowest-SFT task (PnP)**, and shrink to a tie on
   the already-strong OpenDrawer.
2. **Conditioning baked into pretraining.** The paper trains `π_pre` **with** advantage-
   conditioning from the start; we **bolt it on** as a later fine-tune — a weaker integration
   that limits how much the conditioning can shift behavior.
3. **Human corrections — the “C” in RECAP.** The paper includes expert teleoperated
   *interventions* mid-task; our simulation setup has **none**, so we exercise only the
   experience half of the method.
4. **On-policy collection.** The paper **re-collects** fresh data with the latest policy each
   iteration; we largely **reuse off-policy rollouts**. Learning from staler data is the most
   likely cause of the **iter-2 dip** before the iter-3 recovery in the §8 curve.

**Bottom line:** with the two ingredients we isolated — a **scene-aware critic** (§9) and a
**properly tuned ε** (§10) — our faithful re-implementation **clears the strong SFT baseline**
(tie on OpenDrawer, win on PnP). The remaining gap to the paper’s dramatic real-world gains is
a **regime gap**, not a flaw in the method: close the four differences above and RECAP has
more room to pull ahead.

## 12. See it for yourself: rollout videos

Numbers are abstract — let’s watch the policies actually drive the robot. Each clip is
recorded at a **fixed seed (5000)** so SFT and RECAP face the **exact same scene** (no
cherry-picking), with the simulator’s ground-truth success label in the filename.

Below: for each task, the **SFT** rollout and the **RECAP (VLM-critic)** rollout, side by
side in the same scene. **Important:** these are **single fixed-seed samples** (one episode
each), so they show **per-scene luck, not the aggregate** — the real comparison is the §8
table. In particular, the **OpenDrawer** pair below happens to be a scene where **RECAP
misses and SFT succeeds**, even though the two *tie* at 58/58 over 50 episodes — a fair
reminder that one clip proves nothing. We lead with PnP, where RECAP completes the task.

Then a **highlight** (`HIGHLIGHT_PnP_seed5007_*`): a PnP scene where **RECAP succeeds and
SFT fails** — one illustrative episode consistent with RECAP’s small PnP edge in §8 (not, by
itself, proof of a systematic win).

In [ ]:
from pathlib import Path
from IPython.display import Video, display, Markdown
task, disp, rates = 'PickPlaceCounterToCabinet', 'PnP (Counter→Cabinet)', '36 / 38'
g = Path(f'{ASSETS}/videos')
display(Markdown(f'### {disp} — same scene (seed 5000)  ·  §8 success % (SFT / RECAP): {rates}'))
for arm, label in [('sft', 'SFT baseline'), ('recap_vlmvf', 'RECAP (VLM-VF)')]:
    clips = sorted(g.glob(f'{task}_{arm}_seed5000*.mp4'))
    if not clips: print(f'(no {label} clip staged for {task})'); continue
    for p in clips:
        outcome = p.stem.split('seed5000_')[-1]   # 'success' or 'fail'
        display(Markdown(f'**{label}** — {outcome}'))
        display(Video(str(p), embed=True, width=320))

In [ ]:
from pathlib import Path
from IPython.display import Video, display, Markdown
task, disp, rates = 'OpenDrawer', 'OpenDrawer', '58 / 58'
g = Path(f'{ASSETS}/videos')
display(Markdown(f'### {disp} — same scene (seed 5000)  ·  §8 success % (SFT / RECAP): {rates}'))
for arm, label in [('sft', 'SFT baseline'), ('recap_vlmvf', 'RECAP (VLM-VF)')]:
    clips = sorted(g.glob(f'{task}_{arm}_seed5000*.mp4'))
    if not clips: print(f'(no {label} clip staged for {task})'); continue
    for p in clips:
        outcome = p.stem.split('seed5000_')[-1]   # 'success' or 'fail'
        display(Markdown(f'**{label}** — {outcome}'))
        display(Video(str(p), embed=True, width=320))

In [ ]:
from pathlib import Path
from IPython.display import Video, display, Markdown
g = Path(f'{ASSETS}/videos')
display(Markdown('### Highlight — PnP, seed 5007: a scene where **RECAP succeeds and SFT fails**'))
for p in sorted(g.glob('HIGHLIGHT_*.mp4')):
    who = 'RECAP (VLM-VF)' if 'recap' in p.stem else 'SFT baseline'
    display(Markdown(f'**{who}**')); display(Video(str(p), embed=True, width=320))

## 13. Run it yourself (Path 2 — needs a GPU)

Want to reproduce the closed-loop numbers? On a GPU machine you can pull our **published
checkpoints** from the HuggingFace Hub and run the exact eval behind the tables above. Two
steps: first a quick environment self-check, then the eval (gated by `RUN_EVAL`).

### 13a. Environment self-check
RoboCasa closed-loop eval needs headless **EGL + MuJoCo**, which is the fragile part of the
setup. This cell verifies the stack *before* you try to run anything. If a line shows `!!`,
see `README.md` §1 for the fix. (On Path 1 / no GPU, it’s fine for items to be missing.)

In [ ]:
def check():
    import importlib, subprocess
    for m in ['lerobot', 'robosuite', 'robocasa', 'mujoco', 'torch']:
        try: importlib.import_module(m); print(f'  ok  {m}')
        except Exception as e: print(f'  !!  {m}: {e}')
    print('  MUJOCO_GL =', os.environ.get('MUJOCO_GL', '(unset — set to egl for headless)'))
    try: print('  GPU:', subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                              capture_output=True, text=True).stdout.strip() or '(no GPU)')
    except Exception: print('  (nvidia-smi unavailable — expected on a no-GPU machine)')
check()

### 13b. Closed-loop eval
With `RUN_EVAL = True` (set it in the config cell), this downloads a published checkpoint and
runs closed-loop eval. The checkpoints are public under
[`dongjin630`](https://huggingface.co/dongjin630):
`recap-robocasa-<task>-sft` (baseline) and `recap-robocasa-<task>-vlmvf` (RECAP, evaluate
with `Advantage: positive`). Mirrors `README.md` §5.

In [ ]:
# NOTE: the RoboCasa env/task id and the HF repo short-name differ ONLY for PnP:
#   env task id 'PickPlaceCounterToCabinet'  ->  repo short-name 'PnPCounterToCab'.
# Map it explicitly; don't build the repo id naively from the task id.
REPO_NAME = {'OpenDrawer': 'OpenDrawer', 'PickPlaceCounterToCabinet': 'PnPCounterToCab'}
if RUN_EVAL:
    import subprocess
    task = 'OpenDrawer'                                 # try 'PickPlaceCounterToCabinet' too
    # checkpoints live permanently under our HF account 'dongjin630' (not a knob to change)
    policy = f'dongjin630/recap-robocasa-{REPO_NAME[task]}-vlmvf'   # or a local CKPT_ROOT path
    cmd = ['python','recap/scripts/eval_cli_wrap.py',
           f'--policy.path={policy}', '--env.type=robocasa', f'--env.task={task}',
           '--eval.batch_size=1', f'--eval.n_episodes={N_EPISODES}', '--eval.use_async_envs=false',
           '--policy.device=cuda', f'--policy.n_action_steps={N_ACTION_STEPS}', f'--seed={SEED}',
           '--output_dir=outputs/eval/notebook_demo']
    print(' '.join(cmd)); subprocess.run(cmd, check=True)
else:
    print('RUN_EVAL=False — set it True in the config cell, on a GPU node, to run closed-loop eval.')

## 14. Limitations, reproducibility, and credits

### Limitations (read these honestly)
Our headline result — RECAP **matches or beats SFT** (tie on OpenDrawer, +2 on PnP) — comes
with real caveats: it required a **scene-aware critic** *and* a **tuned ε = 0.5** (at the
default ε = 0.3 RECAP underperforms, §10); the PnP margin is **small relative to n = 50
noise**; and the iteration curve is **non-monotonic**. The gains are also bounded by the
**four regime gaps in §11** (scale, conditioning-in-pretraining, missing human corrections,
off-policy data). This is a **faithful but down-scaled** study in simulation on **two tasks**
— it should **not** be read as evidence for or against the paper’s real-world claims, which we
did not (and could not) test.

### Reproducibility
- Every number above is **n = 50 episodes at held-out seed 5000**, with `n_action_steps = 10`.
- Per-task specialists are fine-tuned from a **multi-task behavior-cloning base** (`π_pre`),
  pretrained across several RoboCasa kitchen tasks; we report the two tasks of §3.
- Demonstration data (LeRobotDataset v3, HuggingFace Hub):
  [`pepijn223/robocasa_OpenDrawer`](https://huggingface.co/datasets/pepijn223/robocasa_OpenDrawer)
  and [`pepijn223/robocasa_PickPlaceCounterToCabinet`](https://huggingface.co/datasets/pepijn223/robocasa_PickPlaceCounterToCabinet).
- Base model: **π0.5** — [`lerobot/pi05_base`](https://huggingface.co/lerobot/pi05_base).
- Published checkpoints: [`dongjin630`](https://huggingface.co/dongjin630) —
  [`recap-robocasa-OpenDrawer-vlmvf`](https://huggingface.co/dongjin630/recap-robocasa-OpenDrawer-vlmvf),
  [`recap-robocasa-PnPCounterToCab-vlmvf`](https://huggingface.co/dongjin630/recap-robocasa-PnPCounterToCab-vlmvf)
  (RECAP, ε=0.5) and the matching `-sft` baselines.
- Exact commands and environment pins are in `README.md`; the self-check in §13a flags setup
  problems.

### Credits
- **π\*0.6 / RECAP** — Physical Intelligence, *“π\*0.6: a VLA That Learns From Experience”*
  ([arXiv:2511.14759](https://arxiv.org/abs/2511.14759)).
- Built on **[LeRobot](https://github.com/huggingface/lerobot)** (Apache-2.0) and
  **[RoboCasa](https://robocasa.ai)** (robosuite + MuJoCo).
- Demonstration datasets by **[`pepijn223`](https://huggingface.co/pepijn223)** on the
  HuggingFace Hub.
- This is an independent **educational re-implementation** — not affiliated with Physical
  Intelligence.